In [354]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
# from statsmodels.stats.multicomp import MultiComparison  
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [355]:
def create_crosstab(group):
  return pd.crosstab(group['percCredibility'], group['onlineResponse'])

def classify(row):
  if row['groundTruth'] == False and row['percCredibility'] == 'p.False':
    return 'hit'
  elif row['groundTruth'] == False and row['percCredibility'] != 'p.False':
    return 'miss'
  elif row['groundTruth'] == True and row['percCredibility'] == 'p.True':
    return 'hit'
  else:
    return 'miss'

def read_csv(csv_file):
  df = pd.read_csv(csv_file)
  groupedData = df.groupby(['mainHeadline','groundTruth'])
  crosstab_dict = groupedData.apply(create_crosstab)
  crosstab_dict_wide = crosstab_dict.reset_index().pivot_table(index=['mainHeadline','groundTruth'], columns='percCredibility', values=['onlineNo', 'onlineYes'], fill_value=0)
  crosstab_dict_wide.columns = crosstab_dict_wide.columns.map('|'.join).str.strip('|')
  crosstab_dict_wide['total'] = crosstab_dict_wide.sum(axis=1)
  crosstab_dict_wide['PT'] = (crosstab_dict_wide['onlineNo|percTrue'] + crosstab_dict_wide['onlineYes|percTrue']) / crosstab_dict_wide['total']
  crosstab_dict_wide['PF'] = (1 -  crosstab_dict_wide['PT'])
  crosstab_dict_wide['PCT'] = crosstab_dict_wide['onlineYes|percTrue']/ (crosstab_dict_wide['onlineYes|percTrue']+ crosstab_dict_wide['onlineNo|percTrue'])
  crosstab_dict_wide['PCF'] = crosstab_dict_wide['onlineYes|percFalse']/ (crosstab_dict_wide['onlineYes|percFalse']+ crosstab_dict_wide['onlineNo|percFalse'])
  crosstab_dict_wide.fillna(0, inplace=True)
  confData = df.groupby(['mainHeadline','percCredibility','onlineResponse'])['confidence'].agg(['mean', 'std'])
  confData = confData.reset_index().pivot_table(index=['mainHeadline'], columns=['percCredibility','onlineResponse'], values=['mean', 'std'], fill_value=0)
  confData.columns = confData.columns.map('|'.join).str.strip('|')
  summary = confData.join(crosstab_dict_wide)
  summary.reset_index(inplace=True)
  headlineData = summary.to_dict(orient='records')
  return headlineData  

In [373]:
df = pd.read_csv('3exp3tr_raw.csv')
groupedData = df.groupby(['mainHeadline','groundTruth'])
crosstab_dict = groupedData.apply(create_crosstab)
crosstab_dict_wide = crosstab_dict.reset_index().pivot_table(index=['mainHeadline','groundTruth'], columns='percCredibility', values=['onlineNo', 'onlineYes'], fill_value=0)
crosstab_dict_wide.columns = crosstab_dict_wide.columns.map('|'.join).str.strip('|')
crosstab_dict_wide['total'] = crosstab_dict_wide.sum(axis=1)
crosstab_dict_wide['PT'] = (crosstab_dict_wide['onlineNo|percTrue'] + crosstab_dict_wide['onlineYes|percTrue']) / crosstab_dict_wide['total']
crosstab_dict_wide['PF'] = (1 -  crosstab_dict_wide['PT'])
crosstab_dict_wide['PCT'] = crosstab_dict_wide['onlineYes|percTrue']/ (crosstab_dict_wide['onlineYes|percTrue']+ crosstab_dict_wide['onlineNo|percTrue'])
crosstab_dict_wide['PCF'] = crosstab_dict_wide['onlineYes|percFalse']/ (crosstab_dict_wide['onlineYes|percFalse']+ crosstab_dict_wide['onlineNo|percFalse'])
crosstab_dict_wide.fillna(0, inplace=True)
confData = df.groupby(['mainHeadline','percCredibility','onlineResponse'])['confidence'].agg(['mean', 'std'])
confData = confData.reset_index().pivot_table(index=['mainHeadline'], columns=['percCredibility','onlineResponse'], values=['mean', 'std'], fill_value=0)
confData.columns = confData.columns.map('|'.join).str.strip('|')
summary = confData.join(crosstab_dict_wide)
summary.reset_index(inplace=True)
headlineData = summary.to_dict(orient='records')
summary

,mainHeadline,groundTruth,mean|percFalse|onlineNo,mean|percFalse|onlineYes,mean|percTrue|onlineNo,mean|percTrue|onlineYes,std|percFalse|onlineNo,std|percFalse|onlineYes,std|percTrue|onlineNo,std|percTrue|onlineYes,onlineNo|percFalse,onlineNo|percTrue,onlineYes|percFalse,onlineYes|percTrue,total,PT,PF,PCT,PCF
0,h11,False,65.461538,71.000000,44.500000,68.000000,18.777715,0.000000,15.066519,0.000000,13,4,1,1,19,0.263158,0.736842,0.200000,0.071429
1,h13,False,47.833333,82.000000,56.750000,40.000000,22.416850,0.000000,30.456253,9.899495,12,4,1,2,19,0.315789,0.684211,0.333333,0.076923
2,h15,False,77.153846,60.000000,62.800000,0.000000,21.682736,0.000000,16.346253,0.000000,13,5,1,0,19,0.263158,0.736842,0.000000,0.071429
3,h16,False,71.647059,71.000000,59.000000,0.000000,17.410274,0.000000,0.000000,0.000000,17,1,1,0,19,0.052632,0.947368,0.000000,0.055556
4,h18,False,74.357143,82.500000,57.000000,0.000000,23.302714,24.748737,15.394804,0.000000,14,3,2,0,19,0.157895,0.842105,0.000000,0.125000
5,h19,False,65.214286,78.000000,43.250000,0.000000,24.299980,0.000000,35.255023,0.000000,14,4,1,0,19,0.210526,0.789474,0.000000,0.066667
6,h2,False,65.500000,70.333333,45.166667,0.000000,26.688533,10.598742,25.071232,0.000000,10,6,3,0,19,0.315789,0.684211,0.000000,0.230769
7,h21,True,41.400000,0.000000,46.923077,62.000000,20.231164,0.000000,19.997756,0.000000,5,13,0,1,19,0.736842,0.263158,0.071429,0.000000
8,h22,True,54.000000,0.000000,46.538462,54.666667,30.643107,0.000000,20.577558,30.105371,3,13,0,3,19,0.842105,0.157895,0.187500,0.000000
9,h23,True,44.333333,0.000000,62.769231,70.000000,26.576932,0.000000,20.757143,8.717798,3,13,0,3,19,0.842105,0.157895,0.187500,0.000000


In [376]:
headlineData[0]['PT']


0.2631578947368421

In [377]:
def createSimData(fileName, NoParticipants):  
  headlineData = read_csv(fileName)
  dataframes = []
  for headline in headlineData:
    # get the perceived credibility of the headline for each participant
    trueProb = headline['PT']
    conditionalProbs = [headline['PCF'],headline['PCT']]
    # get the perceptions
    perceptions = np.random.binomial(1, trueProb, NoParticipants)
    # get the online responses based on perceptions
    onlineResponses = []
    for i in perceptions:
      onlineResponses.append(np.random.binomial(1, conditionalProbs[i]))
    # get the confidence ratings from a normal distribution based on the mean and std of the distribution.
    condition_map = {
      (1, 1): {'mean': headline['mean|percTrue|onlineYes'], 'std': headline['std|percTrue|onlineYes']},  
      (0, 0): {'mean': headline['mean|percFalse|onlineNo'], 'std': headline['std|percFalse|onlineNo']},
      (1, 0): {'mean': headline['mean|percTrue|onlineNo'], 'std': headline['std|percTrue|onlineNo']},
      (0, 1): {'mean': headline['mean|percFalse|onlineYes'], 'std': headline['std|percFalse|onlineYes']}
    }
    confidence = []
    for a, b in zip(perceptions, onlineResponses):
      stats = condition_map.get((a, b), None)
      confidence.append(np.random.normal(stats['mean'], stats['std']))
    headlineDf = pd.DataFrame({
      'mainHeadline': headline['mainHeadline'],
      'groundTruth': headline['groundTruth'],
      'onlineResponse': onlineResponses,
      'percCredibility': perceptions,
      'confidence': confidence
    })
    dataframes.append(headlineDf)
  simData = pd.concat(dataframes, axis=0)
  simData['onlineResponse'] = simData['onlineResponse'].replace({1: 'yes', 0: 'no'})
  simData['percCredibility'] = simData['percCredibility'].replace({1: 'p.True', 0: 'p.False'})

  simData['classification'] = simData.apply(classify, axis=1)

  return simData

In [378]:
simData = createSimData('3exp3tr_raw.csv', 50)

In [379]:
simData

,mainHeadline,groundTruth,onlineResponse,percCredibility,confidence,classification
0,h11,False,yes,p.True,68.000000,miss
1,h11,False,yes,p.False,71.000000,hit
2,h11,False,no,p.False,76.655557,hit
3,h11,False,no,p.False,65.812907,hit
4,h11,False,no,p.False,41.038826,hit
...,...,...,...,...,...,...
45,h6,False,yes,p.True,55.253240,miss
46,h6,False,no,p.False,105.938996,hit
47,h6,False,yes,p.True,71.790529,miss
48,h6,False,no,p.False,76.167255,hit


In [380]:
groupedData = simData.groupby(['mainHeadline','groundTruth'])
crosstab_dict = groupedData.apply(create_crosstab)
crosstab_dict_wide = crosstab_dict.reset_index().pivot_table(index=['mainHeadline','groundTruth'], columns='percCredibility', values=['no', 'yes'], fill_value=0)
crosstab_dict_wide.columns = crosstab_dict_wide.columns.map('|'.join).str.strip('|')
crosstab_dict_wide['total'] = crosstab_dict_wide.sum(axis=1)
crosstab_dict_wide

,,no|p.False,no|p.True,yes|p.False,yes|p.True,total
mainHeadline,groundTruth,,,,,
h11,False,37,8,3,2,50
h13,False,32,9,3,6,50
h15,False,34,13,3,0,50
h16,False,45,3,2,0,50
h18,False,33,10,7,0,50
h19,False,40,8,2,0,50
h2,False,29,10,11,0,50
h21,True,12,36,0,2,50
h22,True,6,41,0,3,50


Analysis

In [381]:
# H1 - Out of the corrective comments only a small proportion is given online compared to reinforcing comments.
crosstab = pd.crosstab(simData['percCredibility'], simData['onlineResponse'], margins=True)
# crosstab.reset_index(inplace=True)
crosstab

onlineResponse,no,yes,All
percCredibility,,,
p.False,590,70,660
p.True,450,90,540
All,1040,160,1200


In [340]:
def prop_test(arr):
  chi2, pval, degf, expected = chi2_contingency(arr)
  print("Chi-Square Statistic:", chi2)
  print("p-value:", pval)

In [341]:
prop_test([[crosstab.loc['p.True', 'yes'], crosstab.loc['p.True', 'no']], [crosstab.loc['p.False', 'yes'],crosstab.loc['p.False', 'no']]])

Chi-Square Statistic: 16.388028044766365
p-value: 5.16101809719206e-05


In [342]:
# H2 - Participants will be more discerning when giving corrective comments compared to reinforcing ones.

df_filtered = simData[simData['onlineResponse'] == 'yes']

# Create a contingency table
table = pd.crosstab(df_filtered['percCredibility'], df_filtered['classification'])
table

classification,hit,miss
percCredibility,,
p.False,45,3
p.True,56,11


In [343]:
prop_test([[table.loc['p.True', 'miss'], table.loc['p.True', 'hit']], [table.loc['p.False', 'miss'],table.loc['p.False', 'hit']]])

Chi-Square Statistic: 1.8367494981774302
p-value: 0.1753324042528125


In [344]:
def process_data(trialData):
  # Select relevant columns
  df = trialData[['percCredibility', 'onlineResponse', 'confidence']]

  # Group by percCredibility and onlineResponse
  grouped_data = df.groupby(['percCredibility', 'onlineResponse'])

  # Calculate mean confidence and count occurrences
  results = grouped_data.agg(mConf=('confidence', 'mean'), count=('confidence', 'count'))

  return results

In [345]:
process_data(simData)

mConf  count
percCredibility onlineResponse                  
p.False         no              62.348288    548
                yes             73.034952     48
p.True          no              52.024785    337
                yes             55.932713     67

In [346]:
simData.columns

Index(['mainHeadline', 'groundTruth', 'onlineResponse', 'percCredibility',
       'confidence', 'classification'],
      dtype='object')

In [347]:
def analyze_data(df):

  model = ols( 'confidence ~ C(percCredibility) + C(onlineResponse) + C(percCredibility):C(onlineResponse)', data=df).fit()
  print(sm.stats.anova_lm(model, typ=2))


  # # Perform ANOVA
  # anova_results = model.anova()
  # print("ANOVA Results:\n", anova_results)

  # # Perform Tukey's HSD test
  # mc = MultiComparison(model)
  # tukey_results = mc.tukeyhsd()
  # print("\nTukey's HSD Multiple Comparison Test:\n", tukey_results)

In [348]:
analyze_data(simData)

                                             sum_sq     df          F  \
C(percCredibility)                     29285.583523    1.0  51.323471   
C(onlineResponse)                       4760.689934    1.0   8.343188   
C(percCredibility):C(onlineResponse)    1133.175526    1.0   1.985909   
Residual                              568325.579982  996.0        NaN   

                                            PR(>F)  
C(percCredibility)                    1.521977e-12  
C(onlineResponse)                     3.955196e-03  
C(percCredibility):C(onlineResponse)  1.590814e-01  
Residual                                       NaN  
